# Routing — 01: PG primary · ES INDEX · Parquet BACKUP

**Persona:** Platform engineer wiring a multi-sink ingestion pipeline.


**Goal:** Demonstrate the role-based driver refactor's routing composition:
- PostgreSQL as the **Primary** for WRITE and READ (source of truth).
- Elasticsearch as an **INDEX** target — async search/mirror, fed the raw envelope.
- DuckDB (or the file-sink of choice) as a **BACKUP** target with `fmt=parquet`.

All three live under a single `ItemsRoutingConfig` — every driver here is an
items-tier driver (`items_postgresql_driver`, `items_elasticsearch_private_driver`,
`items_duckdb_driver`). `ItemsRoutingConfig.operations` is a flat dict keyed
by `WRITE`/`READ`/`INDEX`/`SEARCH`/`TRANSFORM`/`BACKUP`; INDEX and BACKUP entries
live alongside WRITE/READ in that same dict (the legacy `metadata.operations`
wrapper was removed when the shape was flattened). Collection-tier routing
(envelope metadata) is a separate `CollectionRoutingConfig` and is left at its
default here — discovery already wires PG + ES for it.

## Architecture

```
           POST /items
               │
        ┌──────▼──────┐
        │   Router    │
        └──────┬──────┘
               │ WRITE (sync)
        ┌──────▼──────┐
        │  PG primary │ ───────► source of truth (READ also here)
        └──────┬──────┘
               │ post-commit notify
       ┌───────┴───────┐
       ▼               ▼
  INDEX (ES)     BACKUP (Parquet)
   async           async
```

In [ ]:
import os
import json
import uuid as _uuid
import time as _t

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL    = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
ADMIN_TOKEN = (
    os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or ""
)

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

CATALOG_ID    = f"rtng01-{_uuid.uuid4().hex[:8]}"
COLLECTION_ID = "events"
print(f"Using catalog={CATALOG_ID} collection={COLLECTION_ID}")


In [ ]:
# Bootstrap: create the ephemeral catalog+collection.

catalog = {"id": CATALOG_ID, "type": "Catalog", "title": "Routing demo", "description": "PG+ES+Parquet fan-out example", "stac_version": "1.1.0", "conformsTo": [], "links": []}
for _ in range(3):
    r = client.post("/stac/catalogs", content=json.dumps(catalog))
    if r.status_code in (200, 201, 409):
        break
assert r.status_code in (200, 201, 409), r.text[:400]

collection = {
    "id": COLLECTION_ID, "type": "Collection",
    "title": "Event stream", "description": "Ingest events fanned out to ES + Parquet",
    "stac_version": "1.1.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [[None, None]]}},
    "license": "proprietary", "keywords": ["events"], "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection),
)
assert r.status_code in (200, 201, 409), r.text[:400]
print("Catalog + collection ready")


## Step 1 — Pin the PostgreSQL driver as the primary

Default `ItemsPostgresqlDriverConfig` — attribute + geometry sidecars are
injected automatically via the registry (no explicit `sidecars` needed for
default VECTOR collections).

In [ ]:
pg_cfg = {
    "class_key": "items_postgresql_driver_config",
    # default VECTOR collection → geometry + attributes sidecars auto-injected
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_postgresql_driver_config",
    content=json.dumps(pg_cfg),
)
assert r.status_code in (200, 201, 204), r.text[:400]


## Step 2 — Wire the private ES driver as an INDEX target

The Elasticsearch **private** driver (`ItemsElasticsearchPrivateDriver`) is a
per-tenant search-tier mirror. There is no separate "hidden" or "obfuscated"
mode — it is just another routing-config driver selection. Registering it
under `operations.INDEX` makes the ReindexWorker fan out on post-commit;
`transformed=false` means the raw envelope (PG's primary shape) is forwarded —
no TRANSFORM chain is applied.

In [ ]:
es_cfg = {
    "class_key": "items_elasticsearch_private_driver_config",
    "target": {"index_prefix": "rtng-"},
}
# Many deployments accept the PUT even when the driver module isn't loaded; the
# _validate_routing_entries apply-handler catches that, with a clearer 400 than
# the raw "module missing" error.
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_elasticsearch_private_driver_config",
    content=json.dumps(es_cfg),
)
print(f"ES config PUT → {r.status_code}")

## Step 3 — Wire DuckDB as a BACKUP target emitting Parquet

DuckDB's BACKUP-role behaviour uses its `EXPORT` capability.  The routing entry
declares `fmt="parquet"` so ingestion events carry the sink into the backup
endpoint (`GET .../backup?format=parquet`).

In [ ]:
duck_cfg = {
    "class_key": "items_duckdb_driver_config",
    "warehouse": "/tmp/duckdb-backups",
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_duckdb_driver_config",
    content=json.dumps(duck_cfg),
)
print(f"DuckDB config PUT → {r.status_code}")


## Step 4 — Build the composite routing config

One `ItemsRoutingConfig` ties it all together — every driver here is items-tier.
`operations` is a flat dict; WRITE/READ are sync on the request path, INDEX and
BACKUP entries live in the same dict and are fanned out async (the ReindexWorker
consumes INDEX, the backup endpoint consumes BACKUP).

Collection-envelope routing (`CollectionRoutingConfig`) is left at its default —
discovery already wires PG + ES for it; no override needed for this demo.

In [ ]:
routing_cfg = {
    "class_key": "items_routing_config",
    "operations": {
        "WRITE": [{"driver_ref": "items_postgresql_driver", "on_failure": "fatal"}],
        "READ":  [{"driver_ref": "items_postgresql_driver"}],
        # INDEX — search mirror; transformed=False means "feed the raw envelope"
        "INDEX": [{
            "driver_ref": "items_elasticsearch_private_driver",
            "transformed": False,
            "on_failure": "warn",
        }],
        # BACKUP — file-sink; fmt binds the sink to the ?format=parquet query.
        "BACKUP": [{
            "driver_ref": "items_duckdb_driver",
            "transformed": False,
            "fmt": "parquet",
            "on_failure": "warn",
        }],
    },
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_routing_config",
    content=json.dumps(routing_cfg),
)
assert r.status_code in (200, 201, 204), r.text[:400]
print("Routing configured: PG primary · ES INDEX · DuckDB BACKUP(fmt=parquet)")

## Step 5 — Verify effective routing (read the resolved config)

The config waterfall merges platform → catalog → collection. The per-plugin
GET already returns the waterfall-resolved value — no /effective sub-path exists.

In [ ]:
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_routing_config"
)
assert r.status_code == 200, r.text[:400]
resolved = r.json()
print(json.dumps(resolved, indent=2)[:1200])

## Step 6 — Write a feature; observe WRITE returns immediately after PG commits

INDEX and BACKUP are async — the user request returns on PG success and the
other sinks catch up on the ReindexWorker / backup endpoint read, respectively.


In [ ]:
feat = {
    "type": "Feature",
    "id": f"evt-{_uuid.uuid4().hex[:8]}",
    "geometry": {"type": "Point", "coordinates": [12.4924, 41.8902]},  # Rome
    "properties": {"event_type": "notebook_demo", "severity": "info"},
}
r = client.post(
    f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    content=json.dumps({"type": "FeatureCollection", "features": [feat]}),
)
assert r.status_code in (200, 201, 204), r.text[:400]
print(f"WRITE → {r.status_code}; PG commit done (ES/DuckDB propagation is async)")


## Teardown

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}")
print(f"DELETE catalog → {r.status_code}")
client.close()
